### Import and Read 

In [0]:
from pyspark.sql.functions import trim, col, when, upper

# Read cust_info from Bronze
df = spark.read.table("`databricks-medallion-lakehouse`.bronze.cust_info")

print("="*70)
print("SILVER: cust_info Transformation")
print("="*70)
print(f"\nBronze rows: {df.count():,}")
print("\nFirst 3 rows (BEFORE cleaning):")
df.show(3, truncate=False)

### Trim Spaces 

In [0]:
# Step 1: Trim leading/trailing spaces from first and last names
df_clean = df.select(
    col("cst_id"),
    col("cst_key"),
    trim(col("cst_firstname")).alias("cst_firstname"),
    trim(col("cst_lastname")).alias("cst_lastname"),
    trim(col("cst_marital_status")).alias("cst_marital_status"),
    trim(col("cst_gndr")).alias("cst_gndr"),
    col("cst_create_date")
)

print("\nFirst 3 rows (AFTER Trim):")
df_clean.show(3, truncate=False)


### Standardize Gender (M/F → Male/Female)

In [0]:
# Step 2: Standardize gender codes (M/F → Male/Female)
# This makes reports readable: "5000 Male customers" instead of "5000 M customers"

df_clean = df_clean.withColumn(
    "cst_gndr", 
    when(upper(col("cst_gndr")) == "M", "Male")
    .when(upper(col("cst_gndr")) == "F", "Female")
    .otherwise("Unknown")  # Handle any unexpected values
)

print("\nGender values (sample):")
df_clean.select("cst_gndr").distinct().show()

print("\nFirst 3 rows (with standardize gender):")
df_clean.show(3, truncate=False)

### Standardize Marital Status (M/S → Married/Single)

In [0]:
# Step 3: Standardize marital status codes (M/S → Married/Single)

df_clean = df_clean.withColumn(
    "cst_marital_status",
    when(upper(col("cst_marital_status")) == "M", "Married")
    .when(upper(col("cst_marital_status")) == "S", "Single")
    .otherwise("Unknown")
)

print("\nMarital status values (sample):")
df_clean.select("cst_marital_status").distinct().show()

print("\nFirst 3 rows (with standardized marital status):")
df_clean.show(3, truncate=False)

### Rename Columns to Professional Names

In [0]:
# Step 4: Rename columns to professional, readable names
df_clean = df_clean.select(
    col("cst_id").alias("customer_id"),
    col("cst_key").alias("customer_key"),
    col("cst_firstname").alias("first_name"),
    col("cst_lastname").alias("last_name"),
    col("cst_marital_status").alias("marital_status"),
    col("cst_gndr").alias("gender"),
    col("cst_create_date").alias("created_date")
)

print("\nFinal schema (SILVER cust_info):")
df_clean.printSchema()

print("\nFirst 3 rows (FINAL):")
df_clean.show(10, truncate=False)

### Deduplicate & Write to Silver 

In [0]:
# Step 5: Deduplicate by customer_id (keep first occurrence)
df_clean = df_clean.dropDuplicates(["customer_id"])

rows_after = df_clean.count()
print(f"\nRows after deduplication: {rows_after:,}")

# Drop old Silver table if it exists (has old schema)
spark.sql(f"DROP TABLE IF EXISTS {silver_table}")

# Step 6: Write to Silver
silver_table = "`databricks-medallion-lakehouse`.silver.cust_info"

df_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(silver_table)

print(f"✅ Written to Silver: {silver_table}")